# RigTech · runner_colab_continuacao — DINOv3 + Transformer (per-crop 3-class)

**Filosofia do ciclo**: o modelo é **instrumento fixo de auditoria de rótulos** — mesmo backbone (DINOv3-ViTL16 satélite, congelado), mesma head (2 layers Transformer sobre patch tokens + mean pool), mesmos hiperparâmetros. O que muda entre ciclos é o **dataset** (correções humanas em `Datasets/DaninhasTreinoClientes/`). Cada rodada produz artefatos versionados (`_c1.pt`, `_c2.pt`, ...) pra medir o quanto as correções melhoraram o rótulo.

**Task**: classificação single-label 3-class por **crop de polígono** — para cada polígono anotado, recorto uma janela 224×224 centrada no centroide e classifico:
- `0` = folha_larga
- `1` = folha_estreita
- `2` = mamona

**Uso do modelo pra achar rótulos ruins**: quando o modelo prediz uma classe diferente do rótulo do anotador, **provavelmente o rótulo está errado**. Também sinaliza crops de baixa confiança (< 0.6) como geometria ambígua/confusa. Ambos viram linhas no CSV de suspeitas pra revisão humana.

**Fluxo**:
1. DINOv3 congelado extrai features de cada crop 224×224 (grid 14×14 = 196 tokens de 1024 dim) → checkpoint por site.
2. Split treino/val: faixa espacial dentro de cada fazenda (20% coluna direita = val, margem de 1 tile no eixo x pelo centroide do polígono). Todas as fazendas contribuem pra treino E val.
3. Head Transformer (2 layers, 8 heads, GELU, norm_first) → mean pool dos tokens → LayerNorm → Dropout → Linear(1024, 3).
4. Loss CrossEntropy com peso por classe (Mamona é rara). AdamW + warmup 2 + cosine 25 épocas, early stop patience=7 em macro-F1.
5. Relatório: accuracy, precision/recall por classe, matriz de confusão.
6. Suspeitas: CSV ordenado por (misclass, low confidence) — cada linha aponta pra um polígono candidato a rerrotular.

**Pré-requisitos no Drive**:
- `MyDrive/Datasets/DaninhasTreinoClientes/{Giasa,DoisRiosFlaviano,Flaviano01,CelsoSTE2,Celso01}/{imagem,daninhas}` — layout já em uso.
- Secret `HF_TOKEN` no Colab (ícone chave) — `dinov3-vitl16-pretrain-sat493m` é gated; aceitar termos em `huggingface.co/facebook/dinov3-vitl16-pretrain-sat493m`.

Runtime → GPU **A100** (T4 funciona, só demora mais).

## 1. Montar Drive e instalar dependências

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip -q install rasterio geopandas scikit-image joblib tqdm scikit-learn shapely transformers torch huggingface_hub

## 2. Imports

In [ ]:
import os
import re
import csv
import math
import hashlib
import inspect
import numpy as np
import rasterio
from rasterio.windows import Window
import geopandas as gpd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoImageProcessor
import joblib
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix, f1_score

## 3. Login no Hugging Face

O checkpoint `dinov3-vitl16-pretrain-sat493m` é **gated**. Aceite os termos na página do modelo e coloque o token no *Secrets* do Colab (ícone chave à esquerda) como `HF_TOKEN`. Sem o secret, cai no login interativo.

In [ ]:
from huggingface_hub import login

_hf_token = None
try:
    from google.colab import userdata
    _hf_token = userdata.get('HF_TOKEN')
except Exception:
    _hf_token = os.environ.get('HF_TOKEN')

if _hf_token:
    login(token=_hf_token)
    print('Login no Hugging Face OK (via HF_TOKEN).')
else:
    login()

## 4. Configuração

**Filosofia do ciclo**: NADA aqui muda entre rodadas — mesma config define o instrumento fixo. Só `CICLO` incrementa pra versionar os artefatos de saída.

In [ ]:
# ---- ciclo (versiona artefatos, nao altera treino) ----
CICLO = 1

BASE = '/content/drive/MyDrive/Datasets/DaninhasTreinoClientes'

# Sites: (nome, imagem, [geojsons_de_daninha]). Sem plantacao aqui -- so precisamos
# dos poligonos de daninha pra recortar. Nome do arquivo geojson determina a classe
# (ver GEOJSON_CLASS_MAP abaixo).
PARES_CONFIG = [
    {
        'nome': 'Giasa',
        'imagem': f'{BASE}/Giasa/imagem/Giasa.tif',
        'geojsons': [
            f'{BASE}/Giasa/daninhas/FolhaLargaGiasa (1).geojson',
            f'{BASE}/Giasa/daninhas/FolhaEstreitaGiasa (1).geojson',
            f'{BASE}/Giasa/daninhas/MamonasGiasa (1).geojson',
        ],
    },
    {
        'nome': 'DoisRiosFlaviano',
        'imagem': f'{BASE}/DoisRiosFlaviano/imagem/DoisRiosFlaviano.tif',
        'geojsons': [
            f'{BASE}/DoisRiosFlaviano/daninhas/FolhaLargaDoisRiosFlaviano.geojson',
            f'{BASE}/DoisRiosFlaviano/daninhas/FolhaEstreitaDoisRiosFlaviano.geojson',
            f'{BASE}/DoisRiosFlaviano/daninhas/MamonasDoisRiosFlaviano.geojson',
        ],
    },
    {
        'nome': 'Flaviano',
        'imagem': f'{BASE}/Flaviano01/imagem/Flaviano01.tif',
        'geojsons': [
            f'{BASE}/Flaviano01/daninhas/FolhaLargaFlaviano-1 (1).geojson',
            f'{BASE}/Flaviano01/daninhas/FolhaEstreitaFlaviano-1 (1).geojson',
            f'{BASE}/Flaviano01/daninhas/MamonasFlaviano-1 (1).geojson',
        ],
    },
    {
        'nome': 'CelsoSTE2',
        'imagem': f'{BASE}/CelsoSTE2/imagem/CelsoSTE2.tif',
        'geojsons': [
            f'{BASE}/CelsoSTE2/daninhas/FolhaLargaCelsoSTE-2 (1).geojson',
            f'{BASE}/CelsoSTE2/daninhas/FolhaEstreitaCelsoSTE-2 (1).geojson',
            f'{BASE}/CelsoSTE2/daninhas/MamonasCelsoSTE-2 (1).geojson',
        ],
    },
    {
        'nome': 'Celso01',
        'imagem': f'{BASE}/Celso01/imagem/Celso01.tif',
        'geojsons': [
            f'{BASE}/Celso01/daninhas/FolhasLargas_Celso_01 (1).geojson',
            f'{BASE}/Celso01/daninhas/FolhaEstreita_Celso_01 (1).geojson',
        ],
    },
]

# Classe deduzida do nome do arquivo geojson (case-insensitive substring). NAO
# reordenar -- reinterpreta silenciosamente todo o dataset.
GEOJSON_CLASS_MAP = [
    ('folhalarga', 0),
    ('folhaslarga', 0),
    ('folhaestreita', 1),
    ('folhasestreita', 1),
    ('mamona', 2),
]
CLASS_NAMES = {0: 'folha_larga', 1: 'folha_estreita', 2: 'mamona'}
N_CLASSES = 3

# ---- DINOv3 (variante satelite) ----
DINO_MODEL = 'facebook/dinov3-vitl16-pretrain-sat493m'
CROP_PIXELS = 224     # tamanho do crop enviado ao DINOv3 (multiplo de PATCH)
PATCH = 16            # patch_size do DINOv3
# grade: 14x14 = 196 tokens por crop

# ---- Amostragem ----
# poligonos com area em pixels menor que isso sao ignorados (ruido/sliver)
MIN_POLYGON_PIXELS = 25
# limita crops por classe por site (pra classes MUITO abundantes tipo folha_larga
# em DoisRiosFlaviano) -- None = sem limite
MAX_CROPS_PER_CLASS_PER_SITE = None

# ---- Split treino/validacao: faixa espacial dentro de CADA imagem ----
VAL_FRACTION = 0.20   # 20% da faixa direita (por centroide do poligono em pixel col)
MARGIN_TILES = 1      # gap de 1 tile de CROP_PIXELS entre treino e val (evita vazamento)
SPLIT_AXIS = 'x'      # 'x' = coluna do centroide; 'y' = linha

# ---- Transformer head ----
TRANSFORMER_LAYERS = 2
NHEAD = 8             # 1024 / 8 = 128 por cabeca
TRANSFORMER_DROPOUT = 0.15
CLASSIFIER_DROPOUT = 0.2

EPOCHS = 25
LR = 1e-4
WEIGHT_DECAY = 1e-4
WARMUP_EPOCHS = 2
BATCH = 64            # crops por batch (leve -- 196 tokens por crop)
GRAD_CLIP_NORM = 1.0
EARLY_STOP_PATIENCE = 7
RANDOM_STATE = 42

# ---- Loss ----
# CLASS_WEIGHTS = None -> calculado por 1/freq no treino (compensa raridade da Mamona)
CLASS_WEIGHTS = None
LABEL_SMOOTHING = 0.05

# ---- Suspeitas ----
SUSPECT_LOW_CONF_THRESHOLD = 0.6   # crops com max_softmax < isso viram suspeita 'low_conf'

# ---- Saidas versionadas por ciclo ----
OUTPUT_MODEL = f'/content/drive/MyDrive/modelo_dinov3_crop3class_c{CICLO}.pt'
REPORT_PATH  = f'/content/drive/MyDrive/relatorio_dinov3_crop3class_c{CICLO}.txt'
SUSPECTS_CSV = f'/content/drive/MyDrive/suspeitas_c{CICLO}.csv'
# Cache das features do DINOv3: NAO depende do ciclo. Se voce mudar geojsons de um
# site, a assinatura do site muda e so ele recalcula.
CHECKPOINT_DIR = '/content/drive/MyDrive/dinov3_crop3class_checkpoint'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print('Ciclo:', CICLO, '| output:', OUTPUT_MODEL)

## 5. Carregar DINOv3 (congelado)

Backbone congelado (`eval`, `no_grad`, na GPU). Normalização lida do `AutoImageProcessor` do checkpoint satélite. `interpolate_pos_encoding=True` pra que o pos_encoding do DINOv3 se adapte ao nosso grid 14×14 (default do pretrain é 14×14 pra 224px — casa exato).

In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Dispositivo:', DEVICE, (torch.cuda.get_device_name(0) if DEVICE == 'cuda' else ''))

proc = AutoImageProcessor.from_pretrained(DINO_MODEL)
DINOV3_MEAN = [float(x) for x in proc.image_mean]
DINOV3_STD  = [float(x) for x in proc.image_std]
print('mean:', DINOV3_MEAN, ' std:', DINOV3_STD)

print('Carregando DINOv3 (congelado):', DINO_MODEL)
model = AutoModel.from_pretrained(DINO_MODEL)
model.eval()
for p in model.parameters():
    p.requires_grad = False
model = model.to(DEVICE)

PATCH_SIZE = model.config.patch_size
NUM_REGISTER_TOKENS = getattr(model.config, 'num_register_tokens', 0)
HIDDEN_SIZE = model.config.hidden_size
print(f'patch_size={PATCH_SIZE} hidden_size={HIDDEN_SIZE} num_register_tokens={NUM_REGISTER_TOKENS}')
assert PATCH_SIZE == PATCH, f'PATCH ({PATCH}) != patch_size do modelo ({PATCH_SIZE})'
assert CROP_PIXELS % PATCH == 0, f'CROP_PIXELS ({CROP_PIXELS}) precisa ser multiplo de PATCH ({PATCH})'

mean_t = torch.tensor(DINOV3_MEAN, device=DEVICE).view(1, 3, 1, 1)
std_t  = torch.tensor(DINOV3_STD,  device=DEVICE).view(1, 3, 1, 1)
SUPPORTS_INTERP = 'interpolate_pos_encoding' in inspect.signature(model.forward).parameters
N_TOKENS = (CROP_PIXELS // PATCH) ** 2
print(f'crop {CROP_PIXELS}x{CROP_PIXELS} -> grid {CROP_PIXELS//PATCH}x{CROP_PIXELS//PATCH} = {N_TOKENS} tokens')

In [ ]:
@torch.no_grad()
def extract_crop_features_batch(crops_uint8_bhw3):
    """crops (B, H, W, 3) uint8. Retorna (B, N_TOKENS, HIDDEN_SIZE) float32 numpy.
    Descarta CLS + register tokens ANTES do reshape pra grade."""
    t = torch.from_numpy(crops_uint8_bhw3.astype(np.float32) / 255.0).permute(0, 3, 1, 2)
    t = t.to(DEVICE)
    t = (t - mean_t) / std_t
    kwargs = {'interpolate_pos_encoding': True} if SUPPORTS_INTERP else {}
    out = model(pixel_values=t, **kwargs)
    tokens = out.last_hidden_state.float().cpu().numpy()          # (B, 1+regs+n_patches, hidden)
    patch_tokens = tokens[:, 1 + NUM_REGISTER_TOKENS:]            # (B, N_TOKENS, hidden)
    if patch_tokens.shape[1] != N_TOKENS:
        raise RuntimeError(f'Esperava {N_TOKENS} patch tokens, vieram {patch_tokens.shape[1]}.')
    return patch_tokens

## 6. Coleta de crops por site (com checkpoint)

Para cada polígono de cada geojson: pega centroide em pixels, recorta janela 224×224 centrada, extrai features DINOv3 (float16 pra caber no Drive), guarda com metadados (site, geojson_source, polygon_id, centroid_lon, centroid_lat, centroid_col, centroid_row, class_label). Se polígono for maior que 224px, vemos só o miolo — decisão validada.

In [ ]:
def class_from_geojson_path(path):
    base = os.path.basename(path).lower()
    base = re.sub(r'[^a-z0-9]', '', base)
    for substr, cls in GEOJSON_CLASS_MAP:
        if substr in base:
            return cls
    raise ValueError(f'Nao consegui deduzir classe do arquivo: {path}')


def load_polys(path, raster_crs):
    if not os.path.exists(path):
        print(f'  aviso: arquivo nao encontrado, pulando: {path}')
        return []
    try:
        gdf = gpd.read_file(path)
    except Exception as e:
        print(f'  aviso: nao consegui ler {path}: {e}')
        return []
    if gdf.crs is not None and gdf.crs != raster_crs:
        gdf = gdf.to_crs(raster_crs)
    polys = []
    for i, g in enumerate(gdf.geometry):
        if g is None or g.is_empty:
            continue
        if not g.is_valid:
            g = g.buffer(0)
        if g.is_empty:
            continue
        polys.append((i, g))
    return polys


def read_centered_crop(src, center_col, center_row):
    """Le um crop CROP_PIXELS x CROP_PIXELS x 3 (uint8) centrado em (center_col, center_row).
    Fora dos limites do raster e' zero-pad (nodata)."""
    half = CROP_PIXELS // 2
    col_off = int(round(center_col - half))
    row_off = int(round(center_row - half))
    win = Window(col_off, row_off, CROP_PIXELS, CROP_PIXELS)
    tile = src.read([1, 2, 3], window=win, boundless=True, fill_value=0)
    return np.moveaxis(tile, 0, -1)   # (H, W, 3) uint8


def collect_site_crops(cfg):
    """Retorna list of dicts, um por crop: {'features': (N_TOKENS, HIDDEN_SIZE) fp16,
    'label': int, 'meta': {...}}. Processa em batches pra usar a GPU eficientemente."""
    print(f"\n=== {cfg['nome']} ===")
    crops_out = []
    with rasterio.open(cfg['imagem']) as src:
        raster_crs = src.crs
        raster_w, raster_h = src.width, src.height

        # 1. coletar TODOS os poligonos com sua classe e centroide em pixels
        pending = []   # (class_label, geojson_src, poly_id, centroid_col, centroid_row, lon, lat)
        for gj in cfg['geojsons']:
            cls = class_from_geojson_path(gj)
            polys = load_polys(gj, raster_crs)
            n_kept = 0
            for poly_id, geom in polys:
                cx, cy = geom.centroid.x, geom.centroid.y
                col, row = ~src.transform * (cx, cy)   # coords -> pixel
                # descarta poligonos totalmente fora do raster
                if not (0 <= col < raster_w and 0 <= row < raster_h):
                    continue
                # descarta slivers minusculos
                minx, miny, maxx, maxy = geom.bounds
                pix_w = abs((maxx - minx) / src.transform.a)
                pix_h = abs((maxy - miny) / src.transform.e)
                if pix_w * pix_h < MIN_POLYGON_PIXELS:
                    continue
                pending.append((cls, os.path.basename(gj), poly_id, col, row, cx, cy))
            print(f'  {os.path.basename(gj)} classe={CLASS_NAMES[cls]}: {len(polys)} poligonos ({len(pending)} acumulados)')

        # opcional: cap por classe
        if MAX_CROPS_PER_CLASS_PER_SITE is not None:
            rng = np.random.default_rng(RANDOM_STATE)
            by_class = {}
            for item in pending:
                by_class.setdefault(item[0], []).append(item)
            capped = []
            for cls, items in by_class.items():
                if len(items) > MAX_CROPS_PER_CLASS_PER_SITE:
                    idx = rng.choice(len(items), size=MAX_CROPS_PER_CLASS_PER_SITE, replace=False)
                    items = [items[i] for i in sorted(idx)]
                capped.extend(items)
            pending = capped

        print(f'  total de crops a processar: {len(pending)}')

        # 2. processa em batches pela GPU
        BATCH_GPU = 32
        for i in tqdm(range(0, len(pending), BATCH_GPU), desc=cfg['nome']):
            batch = pending[i:i + BATCH_GPU]
            crops = np.stack([read_centered_crop(src, item[3], item[4]) for item in batch], axis=0)
            feats = extract_crop_features_batch(crops)   # (B, N_TOKENS, HIDDEN_SIZE) float32
            for j, item in enumerate(batch):
                cls, gj_src, poly_id, col, row, lon, lat = item
                crops_out.append({
                    'features': feats[j].astype(np.float16),
                    'label': cls,
                    'meta': {
                        'site': cfg['nome'],
                        'geojson_source': gj_src,
                        'polygon_id': poly_id,
                        'centroid_col': float(col),
                        'centroid_row': float(row),
                        'centroid_lon': float(lon),
                        'centroid_lat': float(lat),
                    },
                })
            if DEVICE == 'cuda':
                torch.cuda.empty_cache()

    per_class = {c: 0 for c in CLASS_NAMES}
    for c in crops_out:
        per_class[c['label']] += 1
    print(f'  crops por classe: ' + ', '.join(f'{CLASS_NAMES[k]}={v}' for k, v in per_class.items()))
    return crops_out


def config_signature_site(cfg):
    """Assinatura sha256 dos params que afetam a COLETA. Muda -> recalcula."""
    geojson_stats = []
    for gj in cfg['geojsons']:
        if os.path.exists(gj):
            st = os.stat(gj)
            geojson_stats.append((os.path.basename(gj), st.st_size, int(st.st_mtime)))
        else:
            geojson_stats.append((os.path.basename(gj), 0, 0))
    payload = repr((
        'dinov3-crop3class', DINO_MODEL, CROP_PIXELS, PATCH, DINOV3_MEAN, DINOV3_STD,
        MIN_POLYGON_PIXELS, MAX_CROPS_PER_CLASS_PER_SITE, cfg['nome'], geojson_stats,
    ))
    return hashlib.sha256(payload.encode()).hexdigest()[:16]


def collect_site_crops_cached(cfg):
    sig = config_signature_site(cfg)
    ck_path = f"{CHECKPOINT_DIR}/{cfg['nome']}_{sig}.joblib"
    if os.path.exists(ck_path):
        d = joblib.load(ck_path)
        print(f"  (cache: {cfg['nome']} -- {len(d)} crops)")
        return d
    crops = collect_site_crops(cfg)
    joblib.dump(crops, ck_path)
    return crops

## 7. Rodar coleta pra todos os sites

In [ ]:
all_crops = []
for cfg in PARES_CONFIG:
    all_crops.extend(collect_site_crops_cached(cfg))

print(f'\nTotal de crops coletados: {len(all_crops)}')
per_class = {c: 0 for c in CLASS_NAMES}
for c in all_crops:
    per_class[c['label']] += 1
print('Distribuicao geral: ' + ', '.join(f'{CLASS_NAMES[k]}={v}' for k, v in per_class.items()))
assert len(all_crops) > 0, 'Nenhum crop coletado -- confira os paths no Drive.'

## 8. Split treino/validação (faixa espacial dentro de cada fazenda)

Pra cada site, ordena os crops pelo centroide no eixo `SPLIT_AXIS`. Os 20% mais à direita viram val. Uma faixa-gap de `MARGIN_TILES * CROP_PIXELS` (em pixels) entre treino e val é descartada — evita vazamento espacial (DINOv3 usa contexto local; crops muito próximos podem compartilhar signal).

In [ ]:
def split_train_val(all_crops, val_fraction, margin_tiles, axis='x'):
    if axis not in ('x', 'y'):
        raise ValueError(f"SPLIT_AXIS invalido: {axis!r}")
    key = 'centroid_col' if axis == 'x' else 'centroid_row'
    margin_px = margin_tiles * CROP_PIXELS

    train, val = [], []
    by_site = {}
    for c in all_crops:
        by_site.setdefault(c['meta']['site'], []).append(c)

    for site, crops in by_site.items():
        crops_sorted = sorted(crops, key=lambda c: c['meta'][key])
        total = len(crops_sorted)
        if total == 0:
            continue
        n_val_target = int(round(total * val_fraction))
        if n_val_target == 0:
            train.extend(crops_sorted)
            print(f'  {site}: total={total} val=0 (fraction {val_fraction} muito pequena)')
            continue
        # val = os n_val_target mais a direita
        val_start_idx = total - n_val_target
        val_start_pos = crops_sorted[val_start_idx]['meta'][key]
        train_end_pos = val_start_pos - margin_px

        n_tr = n_va = n_gap = 0
        for c in crops_sorted:
            p = c['meta'][key]
            if p >= val_start_pos:
                val.append(c); n_va += 1
            elif p < train_end_pos:
                train.append(c); n_tr += 1
            else:
                n_gap += 1
        print(f'  {site}: total={total} treino={n_tr} val={n_va} gap={n_gap} '
              f'| val_start_{axis}={val_start_pos:.0f}px margem={margin_px}px')
    return train, val


train_crops, val_crops = split_train_val(all_crops, VAL_FRACTION, MARGIN_TILES, SPLIT_AXIS)
print(f'\nTotal treino: {len(train_crops)} | val: {len(val_crops)}')

if CLASS_WEIGHTS is None:
    counts = np.zeros(N_CLASSES, dtype=np.float64)
    for c in train_crops:
        counts[c['label']] += 1
    counts_safe = np.maximum(counts, 1.0)
    weights = counts.sum() / (N_CLASSES * counts_safe)
    weights = np.clip(weights, 0.1, 20.0)
    class_weights = weights.astype(np.float32)
else:
    class_weights = np.array(CLASS_WEIGHTS, dtype=np.float32)
print('Pesos por classe: ' + ', '.join(f'{CLASS_NAMES[i]}={class_weights[i]:.2f}' for i in range(N_CLASSES)))

## 9. Dataset e DataLoader

In [ ]:
class CropFeaturesDataset(Dataset):
    def __init__(self, crops):
        self.crops = crops

    def __len__(self):
        return len(self.crops)

    def __getitem__(self, idx):
        c = self.crops[idx]
        feats = torch.from_numpy(c['features'].astype(np.float32))   # (N_TOKENS, HIDDEN_SIZE)
        label = torch.tensor(c['label'], dtype=torch.long)
        return feats, label


train_loader = DataLoader(
    CropFeaturesDataset(train_crops), batch_size=BATCH, shuffle=True, drop_last=False,
)
val_loader = None
if len(val_crops) > 0:
    val_loader = DataLoader(
        CropFeaturesDataset(val_crops), batch_size=BATCH, shuffle=False, drop_last=False,
    )
print(f'batches treino: {len(train_loader)}' + (f' | batches val: {len(val_loader)}' if val_loader else ''))

## 10. Arquitetura do head

Igual ao notebook base (LayerNorm de entrada + pos_encoding aprendível + 2 layers TransformerEncoder GELU norm_first), mas com **mean pool** no fim pra colapsar os 196 tokens em 1 vetor → Linear(1024, 3).

In [ ]:
class CropTransformerHead(nn.Module):
    def __init__(self, hidden_size, n_tokens, n_layers=2, nhead=8, dropout=0.1,
                 classifier_dropout=0.0, n_classes=3):
        super().__init__()
        # tokens do DINOv3 tem escala bem maior que pos_encoding (std=0.02) --
        # normaliza os tokens antes de somar pos.
        self.input_norm = nn.LayerNorm(hidden_size)
        self.pos_encoding = nn.Parameter(torch.zeros(1, n_tokens, hidden_size))
        nn.init.trunc_normal_(self.pos_encoding, std=0.02)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_size, nhead=nhead, dim_feedforward=hidden_size * 2,
            dropout=dropout, batch_first=True, norm_first=True, activation='gelu',
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(hidden_size)
        self.class_dropout = nn.Dropout(classifier_dropout)
        self.classifier = nn.Linear(hidden_size, n_classes)

    def forward(self, tokens):
        # tokens: (B, N_TOKENS, HIDDEN_SIZE)
        x = self.input_norm(tokens) + self.pos_encoding
        x = self.encoder(x)
        x = self.norm(x)
        x = x.mean(dim=1)          # mean pool sobre tokens -> (B, HIDDEN_SIZE)
        x = self.class_dropout(x)
        return self.classifier(x)   # (B, n_classes)


head = CropTransformerHead(
    hidden_size=HIDDEN_SIZE, n_tokens=N_TOKENS, n_layers=TRANSFORMER_LAYERS,
    nhead=NHEAD, dropout=TRANSFORMER_DROPOUT, classifier_dropout=CLASSIFIER_DROPOUT,
    n_classes=N_CLASSES,
).to(DEVICE)
n_params = sum(p.numel() for p in head.parameters() if p.requires_grad)
print(f'Head transformer: {n_params:,} parametros treinaveis')

## 11. Loss, otimizador, scheduler

In [ ]:
class_weight_t = torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)
loss_fn = nn.CrossEntropyLoss(weight=class_weight_t, label_smoothing=LABEL_SMOOTHING)

optimizer = torch.optim.AdamW(head.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

def lr_lambda(epoch_idx):
    if WARMUP_EPOCHS > 0 and epoch_idx < WARMUP_EPOCHS:
        return (epoch_idx + 1) / WARMUP_EPOCHS
    denom = max(EPOCHS - WARMUP_EPOCHS, 1)
    progress = (epoch_idx - WARMUP_EPOCHS) / denom
    progress = min(max(progress, 0.0), 1.0)
    return 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)


def run_epoch(loader, train=True):
    head.train(mode=train)
    total_loss, n_batches = 0.0, 0
    for tokens, labels in loader:
        tokens = tokens.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        with torch.set_grad_enabled(train):
            logits = head(tokens)
            loss = loss_fn(logits, labels)
        if train:
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(head.parameters(), GRAD_CLIP_NORM)
            optimizer.step()
        total_loss += float(loss.item())
        n_batches += 1
    return total_loss / max(n_batches, 1)


@torch.no_grad()
def evaluate(loader):
    if loader is None:
        return None, None, None
    head.eval()
    all_true, all_pred = [], []
    for tokens, labels in loader:
        tokens = tokens.to(DEVICE, non_blocking=True)
        logits = head(tokens)
        pred = logits.argmax(dim=-1).cpu().numpy()
        all_true.append(labels.numpy())
        all_pred.append(pred)
    y_true = np.concatenate(all_true)
    y_pred = np.concatenate(all_pred)
    macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    return macro_f1, y_true, y_pred

## 12. Loop de treino (early stop em macro-F1)

In [ ]:
best_f1 = -1.0
best_state = None
epochs_no_improve = 0

for epoch in range(1, EPOCHS + 1):
    train_loss = run_epoch(train_loader, train=True)
    val_f1, _, _ = evaluate(val_loader)
    current_lr = optimizer.param_groups[0]['lr']
    msg = f'epoca {epoch}/{EPOCHS} | LR: {current_lr:.2e} | loss treino: {train_loss:.4f}'
    if val_f1 is not None:
        msg += f' | macro-F1 (val): {val_f1:.4f}'
        if val_f1 > best_f1:
            best_f1 = val_f1
            best_state = {k: v.detach().cpu().clone() for k, v in head.state_dict().items()}
            epochs_no_improve = 0
            msg += '  (melhor, salvando)'
        else:
            epochs_no_improve += 1
    else:
        best_state = {k: v.detach().cpu().clone() for k, v in head.state_dict().items()}
    print(msg)
    scheduler.step()
    if val_loader is not None and epochs_no_improve >= EARLY_STOP_PATIENCE:
        print(f'early stop -- {EARLY_STOP_PATIENCE} epocas sem melhora.')
        break

if best_state is not None:
    head.load_state_dict(best_state)
print('Treino concluido.' + (f' Melhor macro-F1 (val): {best_f1:.4f}' if best_f1 >= 0 else ''))

torch.save({
    'head_state_dict': head.state_dict(),
    'hidden_size': HIDDEN_SIZE, 'n_tokens': N_TOKENS, 'n_classes': N_CLASSES,
    'transformer_layers': TRANSFORMER_LAYERS, 'nhead': NHEAD,
    'transformer_dropout': TRANSFORMER_DROPOUT, 'classifier_dropout': CLASSIFIER_DROPOUT,
    'label_smoothing': LABEL_SMOOTHING, 'class_weights': class_weights.tolist(),
    'dino_model': DINO_MODEL, 'dinov3_mean': DINOV3_MEAN, 'dinov3_std': DINOV3_STD,
    'crop_pixels': CROP_PIXELS, 'patch': PATCH, 'num_register_tokens': NUM_REGISTER_TOKENS,
    'class_names': CLASS_NAMES, 'best_val_macro_f1': best_f1, 'ciclo': CICLO,
}, OUTPUT_MODEL)
print(f'Modelo salvo em {OUTPUT_MODEL}')

## 13. Relatório de métricas

In [ ]:
report_lines = [f'=== ciclo {CICLO} | DINOv3+Transformer per-crop 3-class ===']
report_lines.append(f'Split: {VAL_FRACTION:.0%} da faixa "{SPLIT_AXIS}" por site, margem={MARGIN_TILES} tiles ({MARGIN_TILES*CROP_PIXELS}px)')
report_lines.append(f'Total crops: {len(all_crops)} | treino: {len(train_crops)} | val: {len(val_crops)}')
report_lines.append('')

if val_loader is not None:
    val_f1, y_true, y_pred = evaluate(val_loader)
    report_lines.append(f'macro-F1 (val): {val_f1:.4f}')
    accuracy = (y_true == y_pred).mean()
    report_lines.append(f'accuracy (val): {accuracy:.4f}')
    report_lines.append('')
    report_lines.append(classification_report(y_true, y_pred, target_names=[CLASS_NAMES[i] for i in range(N_CLASSES)], zero_division=0))
    report_lines.append('Matriz de confusao [linhas=verdadeiro, colunas=previsto]:')
    report_lines.append('              ' + '  '.join(f'{CLASS_NAMES[i]:>15}' for i in range(N_CLASSES)))
    cm = confusion_matrix(y_true, y_pred, labels=list(range(N_CLASSES)))
    for i, row in enumerate(cm):
        report_lines.append(f'{CLASS_NAMES[i]:>14} ' + '  '.join(f'{v:>15d}' for v in row))
else:
    report_lines.append('sem tiles de validacao -- relatorio de metricas pulado.')

report_text = '\n'.join(report_lines)
print(report_text)
with open(REPORT_PATH, 'w', encoding='utf-8') as f:
    f.write(report_text)
print(f'\nRelatorio salvo em {REPORT_PATH}')

## 14. Exportar suspeitas → CSV pra revisão humana

Passa TODOS os crops (treino + val) pelo modelo. Marca como suspeita:
- **misclass**: `pred != true` — provável rótulo errado
- **low_conf**: `max_softmax < SUSPECT_LOW_CONF_THRESHOLD` — geometria ambígua/confusa

Ordena misclass primeiro (mais informativo), depois low_conf por confidence crescente. Cada linha aponta pro polígono via `site` + `geojson_source` + `polygon_id` + coords do centroide (pra abrir no QGIS).

In [ ]:
@torch.no_grad()
def infer_all(crops):
    head.eval()
    loader = DataLoader(CropFeaturesDataset(crops), batch_size=BATCH, shuffle=False)
    preds, confs, probs_all = [], [], []
    for tokens, _ in loader:
        tokens = tokens.to(DEVICE, non_blocking=True)
        logits = head(tokens)
        probs = torch.softmax(logits, dim=-1).cpu().numpy()
        preds.append(probs.argmax(axis=-1))
        confs.append(probs.max(axis=-1))
        probs_all.append(probs)
    return np.concatenate(preds), np.concatenate(confs), np.concatenate(probs_all, axis=0)

all_preds, all_confs, all_probs = infer_all(all_crops)

suspects = []
for i, c in enumerate(all_crops):
    true = c['label']
    pred = int(all_preds[i])
    conf = float(all_confs[i])
    kind = None
    if pred != true:
        kind = 'misclass'
    elif conf < SUSPECT_LOW_CONF_THRESHOLD:
        kind = 'low_conf'
    if kind is None:
        continue
    suspects.append({
        **c['meta'],
        'true_class': CLASS_NAMES[true],
        'pred_class': CLASS_NAMES[pred],
        'confidence': conf,
        'prob_folha_larga': float(all_probs[i, 0]),
        'prob_folha_estreita': float(all_probs[i, 1]),
        'prob_mamona': float(all_probs[i, 2]),
        'kind': kind,
    })

# ordena: misclass primeiro (por confidence decrescente -- mais confiante = mais provavel erro),
# depois low_conf (por confidence crescente -- pior primeiro)
suspects.sort(key=lambda s: (0 if s['kind'] == 'misclass' else 1,
                             -s['confidence'] if s['kind'] == 'misclass' else s['confidence']))

print(f'Total de suspeitas: {len(suspects)} '
      f'({sum(1 for s in suspects if s["kind"]=="misclass")} misclass, '
      f'{sum(1 for s in suspects if s["kind"]=="low_conf")} low_conf) '
      f'de {len(all_crops)} crops totais')

if suspects:
    fieldnames = ['kind', 'site', 'geojson_source', 'polygon_id', 'true_class', 'pred_class',
                  'confidence', 'prob_folha_larga', 'prob_folha_estreita', 'prob_mamona',
                  'centroid_lon', 'centroid_lat', 'centroid_col', 'centroid_row']
    with open(SUSPECTS_CSV, 'w', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        for s in suspects:
            w.writerow({k: s.get(k, '') for k in fieldnames})
    print(f'Suspeitas salvas em {SUSPECTS_CSV}')
    print('\nTop 10 suspeitas:')
    for s in suspects[:10]:
        print(f"  [{s['kind']}] {s['site']}/{s['geojson_source']}#{s['polygon_id']} "
              f"true={s['true_class']} pred={s['pred_class']} conf={s['confidence']:.3f}")

## 15. Próximo ciclo

1. Abrir `suspeitas_c1.csv` no editor (ou QGIS pelas coords `centroid_lon/lat`) e revisar as manchas marcadas.
2. Corrigir os geojsons no Drive (mover polígono pro arquivo da classe certa, ajustar geometria, deletar sliver).
3. Bump `CICLO = 2` no topo, rodar tudo de novo.
4. **Cache é inteligente**: a assinatura de cada site inclui `mtime` dos geojsons — só os sites cujos geojsons mudaram recalculam features do DINOv3. Os outros voltam do cache em segundos.
5. Comparar `relatorio_c2.txt` vs `relatorio_c1.txt`: macro-F1 subiu? Que classes melhoraram? O número de suspeitas caiu? → dataset melhorou.

**Filosofia**: se o modelo é bem calibrado (mesmo instrumento, mesmos params), qualquer melhora de métrica entre ciclos vem de rótulos melhores — não de tuning.